# 2.2 — Clinical Fine-Tuning on Mendeley
**AIKONIC — Model Training**

This notebook runs:
- **Phase A** — Frozen MobileNetV3-Small base, LR=1e-4, up to 30 epochs
- **Phase B** — Full unfreeze, LR=1e-6, up to 50 epochs

Outputs:
- `checkpoints/phase_a_frozen.h5`
- `checkpoints/production_model_final.h5`
- `training_plots/` — loss/accuracy/sensitivity curves
- `logs/training_metrics.csv`

**Assigned to:** ML Trainer I (JASMINE ROLLON), ML Trainer II (EUNICE JAIMEE LEDESMA)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf

import config
from data_loader import DataLoader
from callbacks import get_phase_a_callbacks, get_phase_b_callbacks

os.makedirs(config.TRAINING_PLOTS_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINTS_DIR,   exist_ok=True)
os.makedirs(config.LOGS_DIR,          exist_ok=True)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────────────────
loader = DataLoader()
counts = loader.get_sample_counts()
print(f'Split counts: {counts}')
loader.validate_no_leakage()

train_ds      = loader.get_train_dataset()
val_ds        = loader.get_val_dataset()
class_weights = loader.get_class_weights()
print(f'Class weights: {class_weights}')

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────────
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Small

def build_model(freeze_base=True, dropout_rate=config.DROPOUT_RATE,
                learning_rate=config.PHASE_A_LR):
    inputs = tf.keras.Input(shape=(224, 224, 1), name='grayscale_input')
    x = layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1), name='channel_replication')(inputs)
    base = MobileNetV3Small(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = not freeze_base
    x = base(x, training=not freeze_base)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.BatchNormalization(name='batch_norm')(x)
    x = layers.Dense(config.DENSE_UNITS, activation=config.ACTIVATION, name='dense_128')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(config.NUM_CLASSES, activation='softmax', name='classifier')(x)
    model = Model(inputs=inputs, outputs=outputs, name='DysgraphiaCNN')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

print('build_model() ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE A — Frozen base fine-tuning
# ══════════════════════════════════════════════════════════════════════════════
print('─' * 60)
print('  PHASE A — Frozen Base Fine-Tuning')
print(f'  LR={config.PHASE_A_LR} | Max epochs={config.PHASE_A_EPOCHS} | Patience={config.PHASE_A_ES_PATIENCE}')
print('─' * 60)

model_a = build_model(freeze_base=True, learning_rate=config.PHASE_A_LR)

callbacks_a = get_phase_a_callbacks(
    checkpoint_path=config.CHECKPOINT_PHASE_A,
    metrics_csv=config.TRAINING_METRICS,
)

history_a = model_a.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.PHASE_A_EPOCHS,
    callbacks=callbacks_a,
    class_weight=class_weights,
    verbose=1,
)

print(f'\n✓ Phase A complete → {config.CHECKPOINT_PHASE_A}')

In [ ]:
# ── Sanity check: accuracy should be above 50% by epoch 5 ─────────────────────
if len(history_a.history['accuracy']) >= 5:
    acc_at_5 = history_a.history['accuracy'][4]
    status = '✓' if acc_at_5 > 0.50 else '⚠'
    print(f'{status} Accuracy at epoch 5: {acc_at_5:.4f} (should be >0.50)')

# ── Plot Phase A curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Phase A Training Curves (Frozen Base)', fontsize=14, fontweight='bold')

for ax, (tr_key, val_key, title) in zip(axes.flatten(), [
    ('loss',      'val_loss',     'Loss'),
    ('accuracy',  'val_accuracy', 'Accuracy'),
    ('recall',    'val_recall',   'Sensitivity (Recall)'),
    ('auc',       'val_auc',      'AUC'),
]):
    if tr_key not in history_a.history: ax.set_visible(False); continue
    ep = range(1, len(history_a.history[tr_key]) + 1)
    ax.plot(ep, history_a.history[tr_key],              label='Train', lw=2)
    ax.plot(ep, history_a.history.get(val_key, []),     label='Val',   lw=2, linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
out_a = os.path.join(config.TRAINING_PLOTS_DIR, 'phase_a_curves.png')
plt.savefig(out_a, dpi=120, bbox_inches='tight'); plt.show()
print(f'Phase A curves saved → {out_a}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE B — Full unfreeze fine-tuning
# ══════════════════════════════════════════════════════════════════════════════
print('─' * 60)
print('  PHASE B — Full Unfreeze Fine-Tuning')
print(f'  LR={config.PHASE_B_LR} | Max epochs={config.PHASE_B_EPOCHS} | Patience={config.PHASE_B_ES_PATIENCE}')
print('─' * 60)

# Load Phase A checkpoint and unfreeze all layers
model_b = tf.keras.models.load_model(config.CHECKPOINT_PHASE_A)
for layer in model_b.layers:
    layer.trainable = True
print('✓ All layers unfrozen')

# Recompile with Phase B LR
model_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.PHASE_B_LR),
    loss='categorical_crossentropy',
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)
print(f'✓ Recompiled with lr={config.PHASE_B_LR}')

In [ ]:
callbacks_b = get_phase_b_callbacks(
    checkpoint_path=config.CHECKPOINT_PROD,
    metrics_csv=config.TRAINING_METRICS,
)

history_b = model_b.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.PHASE_B_EPOCHS,
    callbacks=callbacks_b,
    class_weight=class_weights,
    verbose=1,
)

print(f'\n✓ Phase B complete → {config.CHECKPOINT_PROD}')

In [ ]:
# ── Plot Phase B curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Phase B Training Curves (Full Unfreeze)', fontsize=14, fontweight='bold')

for ax, (tr_key, val_key, title) in zip(axes.flatten(), [
    ('loss',      'val_loss',     'Loss'),
    ('accuracy',  'val_accuracy', 'Accuracy'),
    ('recall',    'val_recall',   'Sensitivity (Recall)'),
    ('auc',       'val_auc',      'AUC'),
]):
    if tr_key not in history_b.history: ax.set_visible(False); continue
    ep = range(1, len(history_b.history[tr_key]) + 1)
    ax.plot(ep, history_b.history[tr_key],          label='Train', lw=2)
    ax.plot(ep, history_b.history.get(val_key, []), label='Val',   lw=2, linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
out_b = os.path.join(config.TRAINING_PLOTS_DIR, 'phase_b_curves.png')
plt.savefig(out_b, dpi=120, bbox_inches='tight'); plt.show()
print(f'Phase B curves saved → {out_b}')

In [ ]:
# ── Combined Phase A + B curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase A + B Combined — Loss & Sensitivity', fontsize=13, fontweight='bold')

for ax, (key, val_key, title) in zip(axes, [
    ('loss',   'val_loss',   'Loss'),
    ('recall', 'val_recall', 'Sensitivity (Recall)'),
]):
    a_len = len(history_a.history.get(key, []))
    b_len = len(history_b.history.get(key, []))
    x_a = range(1, a_len + 1)
    x_b = range(a_len + 1, a_len + b_len + 1)
    ax.plot(x_a, history_a.history.get(key,     []), color='steelblue',  label='Phase A train')
    ax.plot(x_a, history_a.history.get(val_key, []), color='steelblue',  label='Phase A val', linestyle='--')
    ax.plot(x_b, history_b.history.get(key,     []), color='darkorange', label='Phase B train')
    ax.plot(x_b, history_b.history.get(val_key, []), color='darkorange', label='Phase B val', linestyle='--')
    ax.axvline(x=a_len, color='gray', linestyle=':', label='Phase boundary')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
combined = os.path.join(config.TRAINING_PLOTS_DIR, 'phase_ab_combined.png')
plt.savefig(combined, dpi=120, bbox_inches='tight'); plt.show()
print(f'Combined curves saved → {combined}')

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print('=' * 60)
print('  TRAINING COMPLETE')
print('=' * 60)
print(f'  Phase A checkpoint : {config.CHECKPOINT_PHASE_A}')
print(f'  Production model   : {config.CHECKPOINT_PROD}')
print(f'  Training metrics   : {config.TRAINING_METRICS}')
print(f'  Training plots     : {config.TRAINING_PLOTS_DIR}/')
print()
print('  ✓ Notify CNN Architect, SHAP Developers, Integration Lead')